<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/GEMINI_TPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow --upgrade -q

In [6]:
# --- 1. JAX 0.9.0+ AUTO-INITIALIZATION ---
import os, jax
import jax.numpy as jnp
from google import genai
from google.genai import types
from google.colab import userdata

def execute_tpu_humanoid_demo():
    print("🚀 Initializing Google Compute Engine TPU Backend...")

    # In 2026, simply checking jax.devices() triggers the XLA-to-TPU handshake.
    try:
        tpu_devices = jax.devices()
        tpu_kind = tpu_devices[0].device_kind
        print(f"✅ TPU DETECTED: {tpu_kind}")
        print(f"🔢 CHIPS AVAILABLE: {jax.device_count()}\n")

        # PROOF: Matrix multiplication on TPU v6e/v5e
        # Using a size optimized for the 256x256 MXU
        size = 8192
        x = jnp.ones((size, size))
        _ = jnp.dot(x, x).block_until_ready()
        print(f"⚡ HARDWARE PROOF: {size}x{size} MatMul completed on TPU.\n")

    except Exception as e:
        print(f"❌ HARDWARE ERROR: {e}")
        return

    # --- 2. GEMINI 3.1 PRO (DEEP THINK MODE) ---
    print("🧠 Connecting to Gemini 3.1 Pro...")

    # Authenticate using your key named 'GEMINI'
    client = genai.Client(api_key=userdata.get('GEMINI'))

    # Prompting for your H2E/JEPA research
    prompt = "Create a state-space model for a 22-DoF humanoid. Compare JEPA with Traditional RL for holonomic gait transitions."

    # Gemini 3.1 Pro uses 'thinking_level' (NOT thinking_budget)
    # include_thoughts is a boolean inside ThinkingConfig
    response = client.models.generate_content(
        model="gemini-3.1-pro-preview",
        contents=prompt,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(
                include_thoughts=True,
                thinking_level="HIGH"  # Options: "LOW", "MEDIUM", "HIGH"
            )
        )
    )

    # --- 3. OUTPUT PARSING ---
    print("-" * 30)
    for part in response.candidates[0].content.parts:
        if part.thought:
            print(f"💡 INTERNAL REASONING (TPU-Powered):\n{part.text}\n")
        else:
            print(f"🏁 FINAL EXPERT ANALYSIS:\n{part.text}")

# Run the unified workflow
execute_tpu_humanoid_demo()

🚀 Initializing Google Compute Engine TPU Backend...
✅ TPU DETECTED: TPU v6 lite
🔢 CHIPS AVAILABLE: 1

⚡ HARDWARE PROOF: 8192x8192 MatMul completed on TPU.

🧠 Connecting to Gemini 3.1 Pro...
------------------------------
💡 INTERNAL REASONING (TPU-Powered):
**Here's how I'd approach this problem, as an expert in the field:**

First, I need to thoroughly understand the prompt. It's asking for two distinct components: a state-space model for a 22-DoF humanoid robot and a comparison between JEPA and traditional RL for holonomic gait transitions. The key is to be precise in the humanoid's representation and the implications of "holonomic" in the context of a humanoid.

**Part 1: The State-Space Model**

Let's break down the 22 DoFs: I must allocate these logically across a humanoid body. Since the prompt asks for a "22-DoF humanoid," it's safe to assume this represents *actuated* joints. Therefore, the floating base (6 DoFs) will be considered separately. The prompt *does not* specify wheth

## JAX Implementation: JEPA Latent Predictor

In [8]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, random
import time

# --- 1. CONFIGURATION ---
# State: [q(29), v(28)] = 57 dims
# Control: [tau(22)] = 22 dims
STATE_DIM = 57
ACTION_DIM = 22
LATENT_DIM = 128
SEED = 123 # Using your preferred seed

# --- 2. JEPA ARCHITECTURE (FLAX-STYLE FUNCTIONAL) ---
def init_jepa_params(key):
    k1, k2, k3 = random.split(key, 3)
    # Encoder: State (57) -> Latent (128)
    # Predictor: Latent (128) + Action (22) -> Next Latent (128)
    return {
        'enc_w': random.normal(k1, (STATE_DIM, LATENT_DIM)) * 0.1,
        'enc_b': jnp.zeros(LATENT_DIM),
        'pred_w': random.normal(k2, (LATENT_DIM + ACTION_DIM, LATENT_DIM)) * 0.1,
        'pred_b': jnp.zeros(LATENT_DIM)
    }

@jit
def encoder(params, state):
    """Encodes raw robot telemetry into physics-aware latent space."""
    return jnp.tanh(jnp.dot(state, params['enc_w']) + params['enc_b'])

@jit
def predictor(params, z_curr, action):
    """Predicts next latent state based on current latent and torque vector."""
    inputs = jnp.concatenate([z_curr, action], axis=-1)
    return jnp.tanh(jnp.dot(inputs, params['pred_w']) + params['pred_b'])

# --- 3. HARDWARE BENCHMARK (The Trillium Proof) ---
def run_jepa_benchmark(params):
    # Simulate a batch of 1024 parallel 22-DoF humanoid states
    batch_size = 1024
    key = random.PRNGKey(SEED)
    states = random.normal(key, (batch_size, STATE_DIM))
    actions = random.normal(key, (batch_size, ACTION_DIM))

    # Vectorized forward pass using vmap
    # This specifically targets the 256x256 MXU on the TPU v6
    v_encoder = vmap(encoder, in_axes=(None, 0))
    v_predictor = vmap(predictor, in_axes=(None, 0, 0))

    print(f"🚀 Benchmarking JEPA on TPU v6 (Batch Size: {batch_size})...")

    # Warm-up (JIT compilation)
    z = v_encoder(params, states).block_until_ready()
    _ = v_predictor(params, z, actions).block_until_ready()

    # Timing
    start = time.time()
    for _ in range(100):
        z = v_encoder(params, states)
        z_next = v_predictor(params, z, actions)
        jax.block_until_ready(z_next)
    end = time.time()

    avg_time = (end - start) / 100
    print(f"✅ Average Prediction Latency: {avg_time*1000:.3f} ms")
    print(f"📊 Throughput: {batch_size / avg_time:.0f} Humanoid States/Sec")

# Initialize and Run
params = init_jepa_params(random.PRNGKey(SEED))
run_jepa_benchmark(params)

🚀 Benchmarking JEPA on TPU v6 (Batch Size: 1024)...
✅ Average Prediction Latency: 0.894 ms
📊 Throughput: 1145108 Humanoid States/Sec


## Complete JEPA Training Loop (JAX + TPU v6)

In [17]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, random, grad
import time

# --- 1. HYPERPARAMETERS & CONFIG ---
STATE_DIM = 57   # 22-DoF + 6-DoF Base + Velocities
ACTION_DIM = 22  # Actuated Torques
LATENT_DIM = 128
BATCH_SIZE = 2048
LEARNING_RATE = 1e-4
EPOCHS = 500
SEED = 123

# --- 2. MODEL INITIALIZATION ---
def init_jepa_params(key):
    k1, k2, k3 = random.split(key, 3)
    return {
        'enc_w': random.normal(k1, (STATE_DIM, LATENT_DIM)) * jnp.sqrt(2/STATE_DIM),
        'enc_b': jnp.zeros(LATENT_DIM),
        'pred_w': random.normal(k2, (LATENT_DIM + ACTION_DIM, LATENT_DIM)) * jnp.sqrt(2/LATENT_DIM),
        'pred_b': jnp.zeros(LATENT_DIM)
    }

# --- 3. CORE ARCHITECTURE ---
@jit
def encoder(params, state):
    """Encodes 22-DoF state into Latent Space Z."""
    return jax.nn.leaky_relu(jnp.dot(state, params['enc_w']) + params['enc_b'])

@jit
def predictor(params, z_curr, action):
    """Predicts next Z based on current Z and action."""
    inputs = jnp.concatenate([z_curr, action], axis=-1)
    return jax.nn.leaky_relu(jnp.dot(inputs, params['pred_w']) + params['pred_b'])

# --- 4. LOSS FUNCTION (Energy-Based + Regularization) ---
@jit
def jepa_loss_fn(params, state_t, action_t, state_t1):
    # Map to Latent Space
    z_t = vmap(encoder, in_axes=(None, 0))(params, state_t)
    z_t1_actual = vmap(encoder, in_axes=(None, 0))(params, state_t1)

    # Predict Next Latent
    z_t1_pred = vmap(predictor, in_axes=(None, 0, 0))(params, z_t, action_t)

    # Invariance Loss (MSE in Latent Space)
    invariance_loss = jnp.mean(jnp.square(z_t1_actual - z_t1_pred))

    # Variance Regularization (Prevents Collapse)
    # We force the standard deviation of each latent dimension to be near 1.0
    std_z = jnp.sqrt(jnp.var(z_t1_actual, axis=0) + 1e-4)
    variance_loss = jnp.mean(jax.nn.relu(1.0 - std_z))

    return invariance_loss + 0.1 * variance_loss

# --- 5. TRAINING STEP ---
@jit
def train_step(params, s_t, a_t, s_t1):
    loss, grads = jax.value_and_grad(jepa_loss_fn)(params, s_t, a_t, s_t1)
    # Simple SGD with weight decay for stability
    new_params = jax.tree_util.tree_map(
        lambda p, g: p - LEARNING_RATE * g, params, grads
    )
    return new_params, loss

# --- 6. EXECUTION LOOP ---
def run_training():
    key = random.PRNGKey(SEED)
    params = init_jepa_params(key)

    print(f"🚀 Starting Training on {jax.devices()[0].device_kind}...")

    # Generate Synthetic Humanoid Data (Replace with your simulation data)
    key, s_key, a_key, n_key = random.split(key, 4)
    s_t = random.normal(s_key, (BATCH_SIZE, STATE_DIM))
    a_t = random.normal(a_key, (BATCH_SIZE, ACTION_DIM))
    s_t1 = s_t + 0.01 * random.normal(n_key, (BATCH_SIZE, STATE_DIM)) # Dummy transition

    start_time = time.time()
    for epoch in range(1, EPOCHS + 1):
        params, loss = train_step(params, s_t, a_t, s_t1)

        if epoch % 100 == 0:
            print(f"📅 Epoch {epoch} | ⚡ Loss: {loss:.6f}")

    total_time = time.time() - start_time
    print(f"\n✅ Training Complete in {total_time:.2f}s")
    print(f"📈 Avg. Epoch Time: {(total_time/EPOCHS)*1000:.3f} ms")

run_training()

🚀 Starting Training on TPU v6 lite...
📅 Epoch 100 | ⚡ Loss: 1.512280
📅 Epoch 200 | ⚡ Loss: 1.508617
📅 Epoch 300 | ⚡ Loss: 1.504997
📅 Epoch 400 | ⚡ Loss: 1.501258
📅 Epoch 500 | ⚡ Loss: 1.497522

✅ Training Complete in 0.40s
📈 Avg. Epoch Time: 0.801 ms


In [20]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, random, grad
import time

# --- 1. PARAMETERS ---
STATE_DIM, ACTION_DIM, LATENT_DIM = 57, 22, 128
BATCH_SIZE, SEED = 2048, 123
key = random.PRNGKey(SEED)

# --- 2. JEPA MODULES ---
def init_params(key):
    k1, k2 = random.split(key)
    return {
        'enc_w': random.normal(k1, (STATE_DIM, LATENT_DIM)) * jnp.sqrt(2/STATE_DIM),
        'enc_b': jnp.zeros(LATENT_DIM),
        'pred_w': random.normal(k2, (LATENT_DIM + ACTION_DIM, LATENT_DIM)) * jnp.sqrt(2/LATENT_DIM),
        'pred_b': jnp.zeros(LATENT_DIM)
    }

@jit
def encoder(params, x):
    return jax.nn.leaky_relu(jnp.dot(x, params['enc_w']) + params['enc_b'])

@jit
def predictor(params, z, a):
    return jax.nn.leaky_relu(jnp.dot(jnp.concatenate([z, a], -1), params['pred_w']) + params['pred_b'])

# --- 3. TRAINING ENGINE ---
@jit
def loss_fn(params, s_t, a_t, s_t1):
    z_t = vmap(encoder, (None, 0))(params, s_t)
    z_t1_actual = vmap(encoder, (None, 0))(params, s_t1)
    z_t1_pred = vmap(predictor, (None, 0, 0))(params, z_t, a_t)

    # JEPA Energy: Prediction Error + Variance Regularization
    pred_err = jnp.mean(jnp.square(z_t1_actual - z_t1_pred))
    std_z = jnp.sqrt(jnp.var(z_t1_actual, axis=0) + 1e-6)
    var_reg = jnp.mean(jax.nn.relu(1.0 - std_z))
    return pred_err + 0.1 * var_reg

@jit
def update(params, s_t, a_t, s_t1, lr=1e-4):
    l, g = jax.value_and_grad(loss_fn)(params, s_t, a_t, s_t1)
    return jax.tree_util.tree_map(lambda p, grad: p - lr * grad, params, g), l

# --- 4. EXECUTION & VALIDATION ---
params = init_params(key)
# Generate a batch of data
s_t = random.normal(key, (BATCH_SIZE, STATE_DIM))
a_t = random.normal(key, (BATCH_SIZE, ACTION_DIM))
s_t1 = s_t + 0.05 * random.normal(key, (BATCH_SIZE, STATE_DIM)) # Stable gait

print("🚀 Training Expert on TPU v6...")
for i in range(501):
    params, loss = update(params, s_t, a_t, s_t1)
    if i % 100 == 0: print(f"Epoch {i} | Loss: {loss:.6f}")

# --- 5. THE H2E RESILIENCE TEST ---
print("\n🛡️ Running H2E-Holonomic Resilience Test...")
# Test 1: Stable Transition (Expected Low Energy)
energy_stable = loss_fn(params, s_t[:1], a_t[:1], s_t1[:1])

# Test 2: Unstable Transition (Simulated sensor failure / noise)
s_t1_noisy = s_t1[:1] + 2.0 * random.normal(key, (1, STATE_DIM))
energy_unstable = loss_fn(params, s_t[:1], a_t[:1], s_t1_noisy)

print(f"✅ Stable Gait Energy:   {energy_stable:.4f}")
print(f"❌ Unstable Gait Energy: {energy_unstable:.4f}")
print(f"📊 Expert Sensitivity:   {energy_unstable/energy_stable:.1f}x Detection Power")

🚀 Training Expert on TPU v6...
Epoch 0 | Loss: 1.593158
Epoch 100 | Loss: 1.589433
Epoch 200 | Loss: 1.585907
Epoch 300 | Loss: 1.582169
Epoch 400 | Loss: 1.578372
Epoch 500 | Loss: 1.574816

🛡️ Running H2E-Holonomic Resilience Test...
✅ Stable Gait Energy:   1.7939
❌ Unstable Gait Energy: 8.3639
📊 Expert Sensitivity:   4.7x Detection Power
